# DAIC-WOZ Dataset Preprocessing Pipeline

**Objective:** Extract multimodal features (audio + text) from DAIC-WOZ interviews for depression detection.

**Output:** Pickle file containing sequence-based features for all participants.

---

## Pipeline Overview

1. **Transcript Parsing:** Extract participant utterances with timestamps
2. **Utterance Segmentation:** Merge consecutive utterances, split long ones
3. **Audio Feature Extraction:** WavLM [768] + Wav2Vec 2.0 [768]
4. **Dataset Assembly:** Sequence of (audio, text, q_type, timestamp) tuples

---

## Configuration

Edit `config/preprocessing_config.yaml` to customize:
- Data paths
- Audio parameters (sample rate, duration thresholds)
- Model selection (Wav2Vec/WavLM variants)
- Question type taxonomy

In [1]:
import os
#Prevent using TensorFlow this project is based on Torch
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"


In [2]:
import sys
sys.path.append('..')

from preprocessing import DatasetBuilder
import yaml
import logging
import warnings

warnings.filterwarnings('ignore')

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)

logger = logging.getLogger(__name__)

c:\Users\Lenovo\anaconda3\lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


## 1. Load Configuration

In [3]:
# Load config
with open('../config/preprocessing_config.yaml', 'r') as f:
    config = yaml.safe_load(f)

print("Configuration loaded:")
print(f"  - Data root: {config['data']['base_path']}")
print(f"  - Sample rate: {config['audio']['sample_rate']} Hz")
print(f"  - Max utterance duration: {config['audio']['max_utterance_duration']}s")
print(f"  - Merge threshold: {config['audio']['max_pause_for_merge']}s")
print(f"  - Models: {config['models']['wav2vec']['name']}, {config['models']['wavlm']['name']}")

Configuration loaded:
  - Data root: ${DAIC_ROOT:../../data/DAIC-WOZ}
  - Sample rate: 16000 Hz
  - Max utterance duration: 15.0s
  - Merge threshold: 2.0s
  - Models: facebook/wav2vec2-base-960h, microsoft/wavlm-base


## 2. Initializing Dataset Builder

In [ ]:
builder = DatasetBuilder(config)
print("\n✓ Builder initialized successfully")

## 3. Run Preprocessing Pipeline

**Warning:** This may take 30-60 minutes depending on:
- Number of participants (~189)
- GPU availability
- Audio file sizes

In [ ]:
# Set checkpoint interval (save every 10 participants)
checkpoint_interval = config['processing'].get('checkpoint_interval', 10)
dataset = builder.build_dataset(checkpoint_interval=checkpoint_interval)

## 4. Print Statistics

In [ ]:
builder.print_statistics()

## 5. Inspect Sample Data

In [ ]:
sample = builder.get_sample_data(dataset)

if sample:
    pid = sample['participant_id']
    data = sample['data']
    
    print("\n" + "="*70)
    print(f"📋 Sample Data Structure (Participant {pid})")
    print("="*70)
    print(f"\nParticipant-level metadata:")
    print(f"  - Label: {data['label']} ({'Depressed' if data['label'] == 1 else 'Non-depressed'})")
    print(f"  - Number of utterances: {data['num_utterances']}")
    print(f"  - Total interview duration: {data['total_duration']:.2f}s")
    print(f"  - Total speaking time: {data['speaking_time']:.2f}s")
    
    if data['sequence']:
        first_utt = data['sequence'][0]
        print(f"\nFirst utterance structure:")
        print(f"  - WavLM embedding shape: {first_utt['wavlm'].shape}")
        print(f"  - Wav2Vec embedding shape: {first_utt['wav2vec'].shape}")
        print(f"  - Question type: {first_utt['q_type']} (ID: {first_utt['q_type_id']})")
        print(f"  - Timestamp: {first_utt['timestamp']:.2f}s")
        print(f"  - Duration: {first_utt['duration']:.2f}s")
        print(f"  - Text: '{first_utt['text'][:100]}..(truncated)'")

## 6. Save Dataset

In [ ]:
builder.save_dataset(dataset)
print("\n Dataset Saved")

## 7. Verify Saved File

In [ ]:
import pickle
from pathlib import Path

output_path = Path(builder.base_path) / config['data']['output_filename']

print(f"\nSaved file_path: {output_path}")

if output_path.exists():
    size_mb = output_path.stat().st_size / (1024 * 1024)
    print(f"  - File size: {size_mb:.2f} MB")
    
    try:
        with open(output_path, 'rb') as f:
            loaded_dataset = pickle.load(f)
        print(f"  - Number of participants: {len(loaded_dataset)}")
        
    except Exception as e:
        print(f"  - Load error: {e}")

**Dataset Structure:**
```python
dataset = {
    'participant_id': {
        'label': int,  # 0(NORMAL) or 1(DEPRESSED)
        'sequence': [  # List of utterances
            {
                'wavlm': np.ndarray[768],
                'wav2vec': np.ndarray[768],
                'text': str,
                'q_type': str,
                'q_type_id': int,
                'timestamp': float,
                'duration': float
            },
            ...
        ],
        'num_utterances': int,
        'total_duration': float,
        'speaking_time': float
    },
    ...
}
```